In [1]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rapidfuzz import process, fuzz

In [2]:
df = pd.read_csv("G:\VSCODE\Python And Data Sceince\Advance Portfolio Projects\Advance Hybrid Recommedation system\Data collection\\tmdb_dataset.csv")

print(df.shape)
df.head()

(17845, 17)


,id,title,overview,genres,keywords,cast,director,production_companies,release_date,vote_average,vote_count,popularity,language,poster_path,backdrop_path,media_type,tags
0,98,Gladiator,"After the death of Emperor Marcus Aurelius, hi...",Action Drama Adventure,"gladiator rome, italy arena senate roman empir...",Russell Crowe Joaquin Phoenix Connie Nielsen O...,Ridley Scott,Universal Pictures Scott Free Productions Red ...,2000-05-04,8.224,20733,17.0640,en,/wN2xWp1eIwCKOD0BHTcErTBv1Uq.jpg,/jhk6D8pim3yaByu1801kMoxXFaX.jpg,movie,"After the death of Emperor Marcus Aurelius, hi..."
1,19576,One Piece: The Movie,There once was a pirate known as the Great Gol...,Action Animation Adventure Comedy Fantasy,pirate gang treasure hunt pirate shounen anime,Mayumi Tanaka Kazuya Nakai Akemi Okamura Kappe...,Junji Shimizu,Toei Animation,2000-03-04,7.062,385,4.3863,ja,/6QOrTYafoqzXPWLnR2K8129jTHH.jpg,/6OnJoCqeRzoOdzXogm64OD3QGzn.jpg,movie,There once was a pirate known as the Great Gol...
2,4327,Charlie's Angels,The captivating crime-fighting trio who are ma...,Action Comedy Adventure,martial arts undercover agent spy secret agent...,Cameron Diaz Drew Barrymore Lucy Liu Bill Murr...,McG,Columbia Pictures Tall Trees Productions Flowe...,2000-11-02,5.852,4475,7.8821,en,/iHTmZs0BmkwMCYi8rhvMWC5G4EM.jpg,/oq3IOr7QvEmei0eXsXacqaMPqlV.jpg,movie,The captivating crime-fighting trio who are ma...
3,146,"Crouching Tiger, Hidden Dragon",Two warriors in pursuit of a stolen sword and ...,Adventure Drama Action Romance,martial arts kung fu based on novel or book fl...,Chow Yun-Fat Michelle Yeoh Zhang Ziyi Chang Ch...,Ang Lee,Sony Pictures Classics Columbia Pictures Film ...,2000-07-06,7.435,3524,8.0508,zh,/iNDVBFNz4XyYzM9Lwip6atSTFqf.jpg,/nSm9cij9VRrGDoZoS16CPnX0FqK.jpg,movie,Two warriors in pursuit of a stolen sword and ...
4,2024,The Patriot,After proving himself on the field of battle i...,Drama History War Action,daughter mission general southern usa loss of ...,Mel Gibson Heath Ledger Joely Richardson Jason...,Roland Emmerich,Columbia Pictures Mutual Film Company Centropo...,2000-06-28,7.183,4257,6.0698,en,/fWZd815QxUCUcrWQZwUkAp9ljG.jpg,/jSHHUHRcShP6DJbg0H2lO4A8m9x.jpg,movie,After proving himself on the field of battle i...


In [3]:
selected_cols = [
    'title',
    'overview',
    'genres',
    'keywords',
    'cast',
    'director',
    'media_type',
    'language',
    'popularity'   # ✅ IMPORTANT
]

df = df[selected_cols]

In [4]:
df.fillna('', inplace=True)

df.isnull().sum()

title         0
overview      0
genres        0
keywords      0
cast          0
director      0
media_type    0
language      0
popularity    0
dtype: int64

In [5]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

for col in ['title', 'overview', 'genres', 'keywords', 'cast', 'director']:
    df[col] = df[col].apply(clean_text)

In [6]:
def normalize_names(text):
    return "_".join(text.split())

df['cast'] = df['cast'].apply(normalize_names)
df['director'] = df['director'].apply(normalize_names)

In [7]:
def create_features(row):
    return (
        (row['genres'] + " ") * 3 +
        (row['cast'] + " ") * 2 +
        (row['director'] + " ") * 2 +
        (row['keywords'] + " ") * 1 +
        (row['overview'] + " ") * 1
    )

df['combined_features'] = df.apply(create_features, axis=1)
df['combined_features'] = df['combined_features'].apply(clean_text)

In [8]:
def classify_content(row):
    if 'anime' in row['keywords'] or 'animation' in row['genres']:
        return 'anime'
    elif row['language'] == 'ko' and row['media_type'] == 'tv':
        return 'kdrama'
    elif row['media_type'] == 'movie':
        return 'movie'
    else:
        return 'tv'

df['content_type'] = df.apply(classify_content, axis=1)

In [9]:
df['clean_title'] = df['title'].apply(clean_text)

title_to_index = pd.Series(df.index, index=df['clean_title']).to_dict()

In [10]:
alias_map = {
    "demon slayer": ["kimetsu no yaiba"],
    "attack on titan": ["shingeki no kyojin"],
    "one piece": ["one piece the movie"],
    "avtar": ["avatar"]
}

In [11]:
def resolve_title(user_input):
    
    user_input = clean_text(user_input)
    tokens = set(user_input.split())
    
    # Alias
    if user_input in alias_map:
        for alias in alias_map[user_input]:
            alias_clean = clean_text(alias)
            if alias_clean in title_to_index:
                return title_to_index[alias_clean]
    
    # Exact
    if user_input in title_to_index:
        return title_to_index[user_input]
    
    # Fuzzy
    matches = process.extract(
        user_input,
        df['clean_title'],
        scorer=fuzz.token_set_ratio,
        limit=10
    )
    
    best_match = None
    best_score = 0
    
    for match, score, _ in matches:
        match_tokens = set(match.split())
        
        if len(tokens.intersection(match_tokens)) == 0:
            continue
        
        if score > best_score and score >= 70:
            best_match = match
            best_score = score
    
    if best_match:
        return title_to_index[best_match]
    
    return None

In [12]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['combined_features'])

tfidf_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df['combined_features'].tolist(), show_progress_bar=True)

sbert_sim = cosine_similarity(embeddings, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/558 [00:00<?, ?it/s]

In [14]:
hybrid_sim = (0.6 * sbert_sim) + (0.4 * tfidf_sim)

In [15]:
def recommend(title, content_type=None, top_k=10):
    
    idx = resolve_title(title)
    
    if idx is None:
        return ["Title not found ❌"]
    
    scores = list(enumerate(hybrid_sim[idx]))
    
    results = []
    
    for i, score in scores:
        
        if i == idx:
            continue
        
        if content_type and df.iloc[i]['content_type'] != content_type:
            continue
        
        popularity = df.iloc[i]['popularity']
        
        final_score = (0.8 * score) + (0.2 * (popularity / df['popularity'].max()))
        
        results.append((i, final_score))
    
    results = sorted(results, key=lambda x: x[1], reverse=True)[:top_k]
    
    return [df.iloc[i]['title'] for i, _ in results]

In [16]:
print(recommend("gladiator", content_type="movie"))
print(recommend("demon slayer", content_type="anime"))
print(recommend("avtar"))

['gladiator ii', 'spartacus', 'pompeii', 'ben hur', 'the lost legion', 'centurion', 'exodus gods and kings', 'the passion of the christ', 'brutus vs cesar', 'the scorpion king 2 rise of a warrior']
['jujutsu kaisen', 'dragon quest the adventure of dai', 'the heike story', 'tokyo ghoul', 'd gray man', 'bleach', 'gintama', 'berserk the golden age arc memorial edition', 'myriad colors phantom world', 'nura rise of the yokai clan']
['Title not found ❌']


In [17]:
def preprocess_query(query):
    
    query = query.lower()
    
    # remove special chars
    query = re.sub(r'[^a-zA-Z0-9\s]', ' ', query)
    
    # remove extra spaces
    query = re.sub(r'\s+', ' ', query).strip()
    
    # remove common noise words
    stop_words = ["movie", "film", "show", "series", "tv"]
    tokens = query.split()
    tokens = [t for t in tokens if t not in stop_words]
    
    query = " ".join(tokens)
    
    return query

In [18]:
def space_correction(query, titles):
    
    for title in titles:
        if query.replace(" ", "") == title.replace(" ", ""):
            return title
    
    return None

In [19]:
def resolve_title(user_input):
    
    # 🔥 NEW: preprocess query
    user_input = preprocess_query(user_input)
    tokens = set(user_input.split())
    
    # -------------------------
    # 🔥 Space correction
    # -------------------------
    corrected = space_correction(user_input, df['clean_title'])
    if corrected:
        return title_to_index[corrected]
    
    # -------------------------
    # Alias (with fuzzy)
    # -------------------------
    if user_input in alias_map:
        for alias in alias_map[user_input]:
            alias_clean = clean_text(alias)
            
            match, score, _ = process.extractOne(
                alias_clean,
                df['clean_title'],
                scorer=fuzz.token_set_ratio
            )
            
            if score >= 70:
                return title_to_index[match]
    
    # -------------------------
    # Exact match
    # -------------------------
    if user_input in title_to_index:
        return title_to_index[user_input]
    
    # -------------------------
    # Fuzzy matching
    # -------------------------
    matches = process.extract(
        user_input,
        df['clean_title'],
        scorer=fuzz.token_set_ratio,
        limit=10
    )
    
    best_match = None
    best_score = 0
    
    for match, score, _ in matches:
        match_tokens = set(match.split())
        
        if len(tokens.intersection(match_tokens)) == 0:
            continue
        
        if score > best_score and score >= 65:
            best_match = match
            best_score = score
    
    if best_match:
        return title_to_index[best_match]
    
    return None

In [20]:
test_queries = [
    "AVTAR!!!",
    "onepiece",
    "   demon   slayer   ",
    "movie avatar 2009",
    "spiderman",
    "charlies angels",
    "demn slayer"
]

for q in test_queries:
    print(q, "→", recommend(q))

AVTAR!!! → ['avatar the deep dive a special edition of 20 20', 'the rookie', 'stranger things', 'from', 'the walking dead', 'breathe', 'guardians of the galaxy', 'midnight special', 'earth to echo', 'journey 2 the mysterious island']
onepiece → ['Title not found ❌']
   demon   slayer    → ['jujutsu kaisen', 'dragon quest the adventure of dai', 'the heike story', 'tokyo ghoul', 'd gray man', 'bleach', 'gintama', 'berserk the golden age arc memorial edition', 'myriad colors phantom world', 'nura rise of the yokai clan']
movie avatar 2009 → ['avatar the deep dive a special edition of 20 20', 'the rookie', 'stranger things', 'from', 'the walking dead', 'breathe', 'guardians of the galaxy', 'midnight special', 'earth to echo', 'journey 2 the mysterious island']
spiderman → ['spider man 2', 'spider man into the spider verse', 'the amazing spider man 2', 'spider man 2 making the amazing', 'the amazing spider man', 'the rookie', 'marvel s ultimate spider man', 'spider man all roads lead to no 

In [21]:
def preprocess_query(query):
    
    query = query.lower()
    
    # remove special chars
    query = re.sub(r'[^a-zA-Z0-9\s]', ' ', query)
    
    # remove numbers (important for "avatar 2009")
    query = re.sub(r'\d+', '', query)
    
    # remove extra spaces
    query = re.sub(r'\s+', ' ', query).strip()
    
    # remove noise words
    stop_words = ["movie", "film", "show", "series", "tv"]
    tokens = [t for t in query.split() if t not in stop_words]
    
    return " ".join(tokens)

In [22]:
def space_correction(query, titles):
    
    query_joined = query.replace(" ", "")
    
    for title in titles:
        if query_joined == title.replace(" ", ""):
            return title
    
    return None

In [23]:
def resolve_title(user_input):
    
    user_input = preprocess_query(user_input)
    tokens = set(user_input.split())
    
    # -------------------------
    # Space correction
    # -------------------------
    corrected = space_correction(user_input, df['clean_title'])
    if corrected:
        return title_to_index[corrected]
    
    # -------------------------
    # Alias (strong fuzzy)
    # -------------------------
    if user_input in alias_map:
        for alias in alias_map[user_input]:
            alias_clean = clean_text(alias)
            
            match, score, _ = process.extractOne(
                alias_clean,
                df['clean_title'],
                scorer=fuzz.token_set_ratio
            )
            
            if score >= 75:   # 🔥 stricter
                return title_to_index[match]
    
    # -------------------------
    # Exact match
    # -------------------------
    if user_input in title_to_index:
        return title_to_index[user_input]
    
    # -------------------------
    # Fuzzy matching (STRICT)
    # -------------------------
    matches = process.extract(
        user_input,
        df['clean_title'],
        scorer=fuzz.token_set_ratio,
        limit=10
    )
    
    best_match = None
    best_score = 0
    
    for match, score, _ in matches:
        match_tokens = set(match.split())
        
        # must share at least 2 tokens OR strong score
        common = tokens.intersection(match_tokens)
        
        if len(common) < 1:
            continue
        
        if score >= 75 and score > best_score:
            best_match = match
            best_score = score
    
    if best_match:
        return title_to_index[best_match]
    
    return None

In [24]:
def recommend(title, content_type=None, top_k=10):
    
    idx = resolve_title(title)
    
    if idx is None:
        return ["Title not found ❌"]
    
    scores = list(enumerate(hybrid_sim[idx]))
    
    results = []
    
    for i, score in scores:
        
        if i == idx:
            continue
        
        if content_type and df.iloc[i]['content_type'] != content_type:
            continue
        
        # 🔥 NEW: filter weak matches
        if score < 0.25:
            continue
        
        popularity = df.iloc[i]['popularity']
        
        final_score = (0.8 * score) + (0.2 * (popularity / df['popularity'].max()))
        
        results.append((i, final_score))
    
    results = sorted(results, key=lambda x: x[1], reverse=True)[:top_k]
    
    return [df.iloc[i]['title'] for i, _ in results]

In [25]:
alias_map.update({
    "onepiece": ["one piece"],
    "spiderman": ["spider man"],
    "batman": ["the batman"],
    "avtar": ["avatar"],
    "demn slayer": ["demon slayer"],
})

In [26]:
test_queries = [
    "AVTAR!!!",
    "onepiece",
    "   demon   slayer   ",
    "movie avatar 2009",
    "spiderman",
    "charlies angels",
    "demn slayer"
]

for q in test_queries:
    print(q, "→", recommend(q))

AVTAR!!! → ['avatar the deep dive a special edition of 20 20', 'stranger things', 'from', 'the walking dead', 'breathe', 'guardians of the galaxy', 'midnight special', 'earth to echo', 'journey 2 the mysterious island', 'the darkest dawn']
onepiece → ['one piece romance dawn story', 'one piece chopper s kingdom on the island of strange animals', 'one piece heart of gold', 'one piece film gold', 'one piece film z', 'one piece giant mecha soldier of karakuri castle', 'luffy s fall the unexplored region grand adventure in the ocean s navel', 'one piece episode of skypiea', 'one piece 3d2y overcome ace s death luffy s vow to his friends', 'one piece episode of chopper plus bloom in the winter miracle cherry blossom']
   demon   slayer    → ['jujutsu kaisen', 'dragon quest the adventure of dai', 'the heike story', 'tokyo ghoul', 'd gray man', 'bleach', 'gintama', 'berserk the golden age arc memorial edition', 'myriad colors phantom world', 'nura rise of the yokai clan']
movie avatar 2009 → 

In [27]:
print(recommend("demon slayer", content_type="anime"))

['jujutsu kaisen', 'dragon quest the adventure of dai', 'the heike story', 'tokyo ghoul', 'd gray man', 'bleach', 'gintama', 'berserk the golden age arc memorial edition', 'myriad colors phantom world', 'nura rise of the yokai clan']


In [28]:
def classify_content(row):
    
    keywords = row['keywords']
    genres = row['genres']
    language = row['language']
    
    # 🔥 strong anime detection
    if (
        'anime' in keywords or
        'animation' in genres and language == 'ja'
    ):
        return 'anime'
    
    elif language == 'ko' and row['media_type'] == 'tv':
        return 'kdrama'
    
    elif row['media_type'] == 'movie':
        return 'movie'
    
    else:
        return 'tv'

In [29]:
df['content_type'] = df.apply(classify_content, axis=1)

In [30]:
def recommend(title, content_type=None, top_k=10):
    
    idx = resolve_title(title)
    
    if idx is None:
        return ["Title not found ❌"]
    
    input_genres = set(df.iloc[idx]['genres'].split())
    
    scores = list(enumerate(hybrid_sim[idx]))
    
    results = []
    
    for i, score in scores:
        
        if i == idx:
            continue
        
        # content filter
        if content_type and df.iloc[i]['content_type'] != content_type:
            continue
        
        # 🔥 genre overlap boost
        candidate_genres = set(df.iloc[i]['genres'].split())
        genre_overlap = len(input_genres.intersection(candidate_genres))
        
        if genre_overlap == 0:
            continue  # remove unrelated content
        
        popularity = df.iloc[i]['popularity']
        
        final_score = (
            0.7 * score +
            0.2 * (popularity / df['popularity'].max()) +
            0.1 * genre_overlap
        )
        
        results.append((i, final_score))
    
    results = sorted(results, key=lambda x: x[1], reverse=True)[:top_k]
    
    return [df.iloc[i]['title'] for i, _ in results]

In [31]:
print("🎌 Demon Slayer → Anime only:")
print(recommend("demon slayer", content_type="anime"))

🎌 Demon Slayer → Anime only:
['jujutsu kaisen', 'dragon quest the adventure of dai', 'bleach', 'd gray man', 'gintama', 'berserk the golden age arc memorial edition', 'nura rise of the yokai clan', 'jojo s bizarre adventure', 'devilman crybaby', 'mob psycho 100']


In [32]:
recommend("demon slayer", content_type="anime")
recommend("avatar", content_type="movie")
recommend("spiderman", content_type="movie")

['spider man 2',
 'spider man into the spider verse',
 'the amazing spider man 2',
 'spider man 2 making the amazing',
 'the amazing spider man',
 'iron man 2',
 'superman shazam the return of black adam',
 'superman returns',
 'iron man',
 'the incredible hulk']

In [34]:
import os
import pickle
import numpy as np

os.makedirs("artifacts", exist_ok=True)

# TF-IDF
with open("artifacts/tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("artifacts/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

# SBERT embeddings (IMPORTANT)
np.save("artifacts/embeddings.npy", embeddings)

# Data
df.to_pickle("artifacts/data.pkl")

# Mappings
with open("artifacts/title_to_index.pkl", "wb") as f:
    pickle.dump(title_to_index, f)

with open("artifacts/alias_map.pkl", "wb") as f:
    pickle.dump(alias_map, f)

print("✅ Saved without hybrid matrix (production safe)")

✅ Saved without hybrid matrix (production safe)
